# NIDS Master Research Pipeline (Comprehensive)

Notebook ini memecah pipeline jadi langkah detail dari awal sampai akhir, mengikuti pola repo referensi (`Intrusion-Detection-Pipeline` dan `Intrusion-Detection-CICIDS2017`) tapi tetap kompatibel dengan implementasi project ini.

## Tujuan
- Menjalankan proses riset secara bertahap: setup -> audit data -> preprocess -> training -> evaluasi -> baseline -> ringkasan.
- Menutup gap dari notebook lama yang terlalu ringkas.
- Menyediakan checkpoint evaluasi yang mudah diverifikasi.


In [ ]:
# Colab-only mount. Di local cell ini akan otomatis skip.
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Running on local environment (no Colab mount).")


In [ ]:
# Resolve project root (works for local + Colab), lalu optional install dependency.
from pathlib import Path
import os
import subprocess
import sys

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/nids-cnn-lstm-autoencoder"),
    Path("/content/drive/MyDrive/nids-cnn-lstm-autoencoder"),
]

PROJECT_ROOT = None
for c in CANDIDATES:
    if (c / "scripts").exists() and (c / "config.yaml").exists():
        PROJECT_ROOT = c.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root tidak ditemukan. Pastikan notebook dijalankan dari repo nids-cnn-lstm-autoencoder.")

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    print("Dependencies installed.")
else:
    print("INSTALL_DEPS = False (skip install).")


In [ ]:
# Imports + helper utilitas eksekusi command dan IO.
import json
import time
import shlex
import shutil
import yaml
import textwrap
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)


def run_cmd_stream(cmd, cwd=None, check=True):
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    start = time.time()
    print("$", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    duration = time.time() - start
    print(f"[exit={proc.returncode}] duration={duration:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc.returncode


def run_cmds_stream(cmd_list, cwd=None, check=True):
    for cmd in cmd_list:
        run_cmd_stream(cmd, cwd=cwd, check=check)


def read_json(path):
    p = Path(path)
    if not p.exists():
        return None
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_config(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_config(cfg, path="config.yaml"):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

print("Helpers loaded.")


## 0) Run Control dan Safety (Isolated Runner)

Notebook ini tidak lagi mengubah `config.yaml` global.
Semua eksekusi eksperimen memakai `run_id` dari `research/sprint2/run_registry.yaml` dan generated config per-run.


In [ ]:
RUN_PREPROCESS = False
RUN_TRAIN_HYBRID = False
RUN_EVAL_ZERO_SHOT = False
RUN_EVAL_FEW_SHOT = False
RUN_TRAIN_BASELINE = False
RUN_EVAL_BASELINE_ZERO_SHOT = False
RUN_EVAL_BASELINE_FEW_SHOT = False
RUN_SUMMARY_ONLY = False

RUNNER_DRY_RUN = False
RUNNER_SKIP_EXISTING = True
FORCE_REGENERATE_CONFIG = False

REGISTRY_PATH = Path("research/sprint2/run_registry.yaml")
BASE_CONFIG_PATH = Path("config/base.yaml")
GENERATED_CONFIG_DIR = Path("research/sprint2/generated_configs")

RUN_ID_HYBRID_ZERO = "s2_hybrid_zero_seed42_base"
RUN_ID_HYBRID_FEW = "s2_hybrid_few_seed42_tp99_eval"
RUN_ID_BASELINE_ZERO = "s2_baseline_zero_seed42_base"
RUN_ID_BASELINE_FEW = "s2_baseline_few_seed42_tp99_eval"
INSPECT_RUN_ID = RUN_ID_HYBRID_ZERO


def load_registry():
    return load_config(str(REGISTRY_PATH))


REGISTRY = load_registry()
RUN_LOOKUP = {r["run_id"]: r for r in REGISTRY["runs"]}


def model_subdir(model_variant: str) -> str:
    return "cnn_lstm_ae" if str(model_variant).lower() == "hybrid" else "lstm_ae"


def generated_config_path(run_id: str) -> Path:
    return GENERATED_CONFIG_DIR / f"{run_id}.yaml"


def ensure_generated_config(run_id: str) -> Path:
    cfg_path = generated_config_path(run_id)
    if cfg_path.exists() and not FORCE_REGENERATE_CONFIG:
        return cfg_path

    cmd = [
        sys.executable,
        "scripts/research_sprint2.py",
        "--registry",
        str(REGISTRY_PATH),
        "--base-config",
        str(BASE_CONFIG_PATH),
        "--run-ids",
        run_id,
        "--dry-run",
        "--no-summarize",
    ]
    run_cmd_stream(cmd, cwd=PROJECT_ROOT)
    if not cfg_path.exists():
        raise FileNotFoundError(f"Generated config not found: {cfg_path}")
    return cfg_path


def resolve_model_path(run_id: str) -> Path:
    run = RUN_LOOKUP[run_id]
    if run.get("model_path"):
        return Path(run["model_path"])

    source_run_id = run.get("reuse_artifacts_from", run_id)
    variant = run.get("model_variant", "hybrid")
    return Path("models") / "research" / source_run_id / model_subdir(variant) / "best_model.keras"


def resolve_data_source_run(run_id: str) -> str:
    run = RUN_LOOKUP[run_id]
    return run.get("reuse_artifacts_from", run_id)


def run_eval_for_run_id(run_id: str):
    cfg_path = ensure_generated_config(run_id)
    run = RUN_LOOKUP[run_id]
    tag = run.get("tag", run_id)
    model_path = resolve_model_path(run_id)

    run_cmd_stream(
        [
            sys.executable,
            "scripts/eval_metrics.py",
            "--config",
            str(cfg_path),
            "--model",
            str(model_path),
            "--tag",
            tag,
        ],
        cwd=PROJECT_ROOT,
    )


def collect_metric_row(run_id: str, variant_label: str):
    run = RUN_LOOKUP[run_id]
    tag = run.get("tag", run_id)
    metrics_root = Path("results") / "research" / run_id / "metrics"

    cic = read_json(metrics_root / f"{tag}_cic_metrics.json")
    cse = read_json(metrics_root / f"{tag}_cse_metrics.json")
    gap = read_json(metrics_root / f"{tag}_generalization_gap.json")

    if not cic or not cse:
        return None

    return {
        "variant": variant_label,
        "run_id": run_id,
        "tag": tag,
        "cic_f1": cic.get("f1"),
        "cse_f1": cse.get("f1"),
        "cic_auc": cic.get("roc_auc"),
        "cse_auc": cse.get("roc_auc"),
        "cic_fpr": cic.get("fpr"),
        "cse_fpr": cse.get("fpr"),
        "f1_gap": None if not gap else gap.get("f1_gap"),
        "accuracy_gap": None if not gap else gap.get("accuracy_gap"),
        "threshold_method": None if not gap else gap.get("threshold_method"),
        "mode": None if not gap else gap.get("mode"),
    }


print("Run controls initialized (isolated runner mode).")
print(f"Registry: {REGISTRY_PATH}")
print(f"Base config: {BASE_CONFIG_PATH}")


## 1) Checklist Adopsi dari Repo Referensi

Cell ini merangkum step detail yang biasanya muncul di repo referensi, lalu dipetakan ke stage notebook ini.


In [ ]:
reference_steps = [
    ("Dataset inventory + schema check", "Section 2.0 - 2.2"),
    ("Duplicate / missing / inf diagnostics", "Section 2.3"),
    ("Label harmonization + feature intersection", "Section 2.4"),
    ("Config review sebelum preprocess", "Section 3.0"),
    ("Preprocess execution + artifact verification", "Section 3.1 - 3.2"),
    ("Training diagnostics (history + checkpoints)", "Section 4.0 - 4.2"),
    ("Zero-shot vs few-shot evaluation", "Section 5.0 - 5.2"),
    ("Baseline comparison", "Section 6.0 - 6.1"),
    ("Research summary export", "Section 7.0"),
]

checklist_df = pd.DataFrame(reference_steps, columns=["reference_pattern", "implemented_in_notebook"])
display(checklist_df)


## 2) Data Discovery dan Pre-Evaluation Audit


In [ ]:
from scripts.utils import list_csv_files

cfg = load_config(str(BASE_CONFIG_PATH if BASE_CONFIG_PATH.exists() else Path("config.yaml")))
paths_cfg = cfg["paths"]

cic_dir = Path(paths_cfg["data_raw_cic"])
cse_dir = Path(paths_cfg["data_raw_cse"])

cic_files = list_csv_files(cic_dir)
cse_files = list_csv_files(cse_dir)


def file_inventory(files, dataset_name):
    rows = []
    total_size = 0
    for p in files:
        size = p.stat().st_size
        total_size += size
        rows.append(
            {
                "dataset": dataset_name,
                "file": p.name,
                "size_mb": round(size / (1024**2), 2),
            }
        )
    return rows, total_size

rows_cic, size_cic = file_inventory(cic_files, "CIC-IDS2017")
rows_cse, size_cse = file_inventory(cse_files, "CSE-CIC-IDS2018")
inventory_df = pd.DataFrame(rows_cic + rows_cse)

print(f"CIC files: {len(cic_files)} | total size: {size_cic/(1024**3):.2f} GB")
print(f"CSE files: {len(cse_files)} | total size: {size_cse/(1024**3):.2f} GB")
display(inventory_df)


### 2.1 Schema Snapshot dan Label Detection

Langkah ini memastikan kita tahu kolom label, kolom yang dibuang, dan perkiraan jumlah fitur mentah.


In [ ]:
from scripts.preprocess import load_header_columns
from scripts.utils import detect_label_column

label_candidates = cfg["preprocess"]["label_candidates"]
drop_columns = set(cfg["preprocess"]["drop_columns"])

summary_rows = []
for dataset_name, files in [("CIC", cic_files), ("CSE", cse_files)]:
    if not files:
        continue
    headers = load_header_columns(files[0])
    label_col = detect_label_column(headers, label_candidates)
    feature_candidates = [c for c in headers if c not in drop_columns and c != label_col]
    summary_rows.append(
        {
            "dataset": dataset_name,
            "sample_file": files[0].name,
            "total_columns": len(headers),
            "detected_label": label_col,
            "feature_candidates_after_drop": len(feature_candidates),
        }
    )

schema_df = pd.DataFrame(summary_rows)
display(schema_df)


### 2.2 Sample Quality Audit (Duplicate, Missing, Inf, Label Mix)

Audit ini mengimitasi langkah EDA dari repo referensi tanpa harus load semua data ke memory.


In [ ]:
def quick_quality_report(csv_path, label_col, sample_rows=50000):
    df = pd.read_csv(csv_path, nrows=sample_rows, skipinitialspace=True)
    df.columns = [str(c).strip() for c in df.columns]

    numeric_df = df.select_dtypes(include=[np.number])
    missing_cells = int(df.isna().sum().sum())
    inf_cells = int(np.isinf(numeric_df.to_numpy()).sum()) if numeric_df.shape[1] > 0 else 0

    row = {
        "file": csv_path.name,
        "rows_sampled": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": missing_cells,
        "inf_cells": inf_cells,
    }

    if label_col in df.columns:
        vc = df[label_col].astype(str).value_counts().head(5).to_dict()
        row["top_labels"] = vc
    else:
        row["top_labels"] = "label_not_found"

    return row

reports = []
for files, dataset_name in [(cic_files, "CIC"), (cse_files, "CSE")]:
    if not files:
        continue
    headers = load_header_columns(files[0])
    label_col = detect_label_column(headers, label_candidates)
    for p in files[:2]:
        rep = quick_quality_report(p, label_col=label_col, sample_rows=50000)
        rep["dataset"] = dataset_name
        reports.append(rep)

quality_df = pd.DataFrame(reports)
display(quality_df)


### 2.3 Feature Intersection dan Harmonisasi CIC vs CSE

Langkah ini mengikuti fungsi preprocessing project (`compute_feature_intersection`, `build_column_mapper`) untuk memastikan alignment sama dengan pipeline utama.


In [ ]:
from scripts.preprocess import compute_feature_intersection, build_column_mapper

cic_label_col, cic_features = compute_feature_intersection(cic_files, label_candidates, list(drop_columns))

cic_reference_cols = load_header_columns(cic_files[0]) if cic_files else []
cse_mapper = {}
for cse_path in cse_files:
    cse_cols = load_header_columns(cse_path)
    cse_mapper.update(build_column_mapper(cic_reference_cols, cse_cols))

cse_label_col, cse_features = compute_feature_intersection(
    cse_files,
    label_candidates,
    list(drop_columns),
    column_mapper=cse_mapper,
)

shared_features = sorted(set(cic_features).intersection(cse_features))

alignment_report = {
    "cic_label_col": cic_label_col,
    "cse_label_col": cse_label_col,
    "cic_feature_count": len(cic_features),
    "cse_feature_count": len(cse_features),
    "shared_feature_count": len(shared_features),
    "mapper_entries": len(cse_mapper),
}

print(json.dumps(alignment_report, indent=2))
print("Sample shared features:", shared_features[:12])


## 3) Preprocessing Stage (Granular)


In [ ]:
cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
cfg = load_config(str(cfg_path))
pre_cfg = cfg["preprocess"]

pre_view = {
    "run_id": RUN_ID_HYBRID_ZERO,
    "generated_config": str(cfg_path),
    "window_size": pre_cfg.get("window_size"),
    "stride": pre_cfg.get("stride"),
    "scaler": pre_cfg.get("scaler"),
    "scaler_fit_mode": pre_cfg.get("scaler_fit_mode"),
    "chunksize": pre_cfg.get("chunksize"),
    "shard_enable": pre_cfg.get("shard_enable"),
    "shard_size": pre_cfg.get("shard_size"),
    "split_by_file": pre_cfg.get("split_by_file"),
    "feature_filter": pre_cfg.get("feature_filter"),
    "scale_guard": pre_cfg.get("scale_guard"),
}
print(json.dumps(pre_view, indent=2))


### 3.1 Execute Preprocess Pipeline

Script preprocess dipanggil dengan generated config per-run:
`research/sprint2/generated_configs/<run_id>.yaml`



In [ ]:
if RUN_PREPROCESS:
    cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
    run_cmd_stream([sys.executable, "scripts/preprocess.py", "--config", str(cfg_path)], cwd=PROJECT_ROOT)
else:
    print("RUN_PREPROCESS=False -> skip preprocess execution.")


### 3.2 Verify Preprocess Artifacts

Validasi manifest shard, scaler, feature metadata, dan report optional.


In [ ]:
data_source_run_id = resolve_data_source_run(INSPECT_RUN_ID)
processed_dir = Path("data") / "research" / data_source_run_id / "processed"
shard_root = processed_dir / "shards"

artifact_paths = {
    "scaler": processed_dir / "scaler.pkl",
    "feature_columns": processed_dir / "feature_columns.json",
    "feature_filter_report": processed_dir / "feature_filter_report.json",
    "scale_guard_report": processed_dir / "scale_guard_report.json",
    "cic_train_manifest": shard_root / "cic" / "train" / "manifest.json",
    "cic_val_manifest": shard_root / "cic" / "val" / "manifest.json",
    "cic_test_manifest": shard_root / "cic" / "test" / "manifest.json",
    "cse_test_manifest": shard_root / "cse" / "test" / "manifest.json",
}

artifact_df = pd.DataFrame(
    [{"artifact": k, "path": str(v), "exists": v.exists()} for k, v in artifact_paths.items()]
)
print(f"Inspect run_id={INSPECT_RUN_ID} | data_source_run_id={data_source_run_id}")
display(artifact_df)

manifest_keys = ["cic_train_manifest", "cic_val_manifest", "cic_test_manifest", "cse_test_manifest"]
manifest_rows = []
for key in manifest_keys:
    m = read_json(artifact_paths[key])
    if m is None:
        continue
    manifest_rows.append(
        {
            "manifest": key,
            "total_samples": m.get("total_samples"),
            "num_shards": m.get("num_shards"),
            "input_shape": m.get("input_shape"),
        }
    )

if manifest_rows:
    display(pd.DataFrame(manifest_rows))
else:
    print("Manifest belum tersedia. Jalankan preprocess dulu.")


### 3.3 Leakage Guard Quick Checks

Validasi cepat terhadap shard split:
- train/val manifest harus ada dan jumlah sample > 0
- test shard harus punya label `y`
- train/val shard memang tanpa `y` (sesuai desain unsupervised training)


In [ ]:
issues = []

train_manifest = read_json(artifact_paths["cic_train_manifest"])
val_manifest = read_json(artifact_paths["cic_val_manifest"])
cic_test_manifest = read_json(artifact_paths["cic_test_manifest"])

if train_manifest is None or train_manifest.get("total_samples", 0) <= 0:
    issues.append("train manifest missing or empty")
if val_manifest is None or val_manifest.get("total_samples", 0) <= 0:
    issues.append("val manifest missing or empty")


def inspect_npz_keys(manifest_obj, label):
    if not manifest_obj or not manifest_obj.get("shards"):
        return None
    first_rel = manifest_obj["shards"][0]["path"]
    npz_path = shard_root / first_rel
    arr = np.load(npz_path)
    return label, list(arr.keys()), {k: arr[k].shape for k in arr.files}

checks = []
for name, mani in [
    ("train", train_manifest),
    ("val", val_manifest),
    ("cic_test", cic_test_manifest),
]:
    out = inspect_npz_keys(mani, name)
    if out is not None:
        checks.append(out)

for label, keys, shapes in checks:
    print(f"{label}: keys={keys}")
    print(f"{label}: shapes={shapes}")

if issues:
    print("Issues:")
    for x in issues:
        print("-", x)
else:
    print("Leakage guard quick checks passed.")


## 4) Hybrid CNN-LSTM AE Training


In [ ]:
cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
cfg = load_config(str(cfg_path))
train_cfg = cfg["training"]

train_view = {
    "run_id": RUN_ID_HYBRID_ZERO,
    "generated_config": str(cfg_path),
    "epochs": train_cfg.get("epochs"),
    "batch_size": train_cfg.get("batch_size"),
    "learning_rate": train_cfg.get("learning_rate"),
    "early_stopping_patience": train_cfg.get("early_stopping_patience"),
    "lr_scheduler": train_cfg.get("lr_scheduler"),
    "cnn_filters": train_cfg.get("cnn_filters"),
    "lstm_units": train_cfg.get("lstm_units"),
    "latent_dim": train_cfg.get("latent_dim"),
    "dropout": train_cfg.get("dropout"),
}
print(json.dumps(train_view, indent=2))


In [ ]:
if RUN_TRAIN_HYBRID:
    cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
    run_cmd_stream([sys.executable, "scripts/train_cnn_lstm_ae.py", "--config", str(cfg_path)], cwd=PROJECT_ROOT)
else:
    print("RUN_TRAIN_HYBRID=False -> skip hybrid training.")


### 4.1 Training History Inspection

Membaca `results/logs/cnn_lstm_history.json` untuk melihat konvergensi.


In [ ]:
history_path = Path("results") / "research" / RUN_ID_HYBRID_ZERO / "logs" / "cnn_lstm_history.json"
history = read_json(history_path)

if history is None:
    print(f"History file not found: {history_path}")
else:
    hist_df = pd.DataFrame(history)
    display(hist_df.tail())

    plot_cols = [c for c in ["loss", "val_loss"] if c in hist_df.columns]
    if plot_cols:
        ax = hist_df[plot_cols].plot(figsize=(10, 4), title="Hybrid Training Loss Curves")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        plt.show()


## 5) Evaluation: Zero-Shot vs Few-Shot


In [ ]:
print("Eval helper ready. Use run_eval_for_run_id(run_id).")


In [ ]:
if RUN_EVAL_ZERO_SHOT:
    run_eval_for_run_id(RUN_ID_HYBRID_ZERO)
else:
    print("RUN_EVAL_ZERO_SHOT=False -> skip hybrid zero-shot eval.")

if RUN_EVAL_FEW_SHOT:
    run_eval_for_run_id(RUN_ID_HYBRID_FEW)
else:
    print("RUN_EVAL_FEW_SHOT=False -> skip hybrid few-shot eval.")


### 5.1 Compare Hybrid Evaluation Variants


In [ ]:
rows = []
for run_id, label in [
    (RUN_ID_HYBRID_ZERO, "hybrid_zero_shot"),
    (RUN_ID_HYBRID_FEW, "hybrid_few_shot"),
]:
    row = collect_metric_row(run_id, label)
    if row:
        rows.append(row)

if not rows:
    print("No hybrid metric files found yet.")
else:
    hybrid_df = pd.DataFrame(rows)
    display(hybrid_df)

    plot_df = hybrid_df[["variant", "cic_f1", "cse_f1"]].set_index("variant")
    plot_df.plot(kind="bar", figsize=(9, 4), title="Hybrid F1 Comparison (CIC vs CSE)")
    plt.ylabel("F1")
    plt.ylim(0, 1)
    plt.show()


## 6) Baseline LSTM Autoencoder + Comparison


In [ ]:
if RUN_TRAIN_BASELINE:
    cfg_path = ensure_generated_config(RUN_ID_BASELINE_ZERO)
    run_cmd_stream([sys.executable, "scripts/train_lstm_ae.py", "--config", str(cfg_path)], cwd=PROJECT_ROOT)
else:
    print("RUN_TRAIN_BASELINE=False -> skip baseline training.")

if RUN_EVAL_BASELINE_ZERO_SHOT:
    run_eval_for_run_id(RUN_ID_BASELINE_ZERO)
else:
    print("RUN_EVAL_BASELINE_ZERO_SHOT=False -> skip baseline zero-shot eval.")

if RUN_EVAL_BASELINE_FEW_SHOT:
    run_eval_for_run_id(RUN_ID_BASELINE_FEW)
else:
    print("RUN_EVAL_BASELINE_FEW_SHOT=False -> skip baseline few-shot eval.")


### 6.1 Hybrid vs Baseline Table


In [ ]:
compare_rows = []
for run_id, label in [
    (RUN_ID_HYBRID_ZERO, "hybrid_zero_shot"),
    (RUN_ID_HYBRID_FEW, "hybrid_few_shot"),
    (RUN_ID_BASELINE_ZERO, "baseline_zero_shot"),
    (RUN_ID_BASELINE_FEW, "baseline_few_shot"),
]:
    row = collect_metric_row(run_id, label)
    if row:
        compare_rows.append(row)

if not compare_rows:
    print("No metrics available for comparison yet.")
else:
    compare_df = pd.DataFrame(compare_rows)
    display(compare_df.sort_values(by=["variant"]))

    view_cols = ["variant", "cic_f1", "cse_f1", "f1_gap", "cic_auc", "cse_auc"]
    plot_ready = compare_df[view_cols].copy()
    display(plot_ready)


## 7) Export Ringkasan Eksekusi Notebook

Cell ini membuat ringkasan markdown yang bisa langsung dipakai untuk dokumentasi sprint/report.


In [ ]:
summary_path = Path("results/research/sprint2/notebook_execution_summary.md")
summary_path.parent.mkdir(parents=True, exist_ok=True)

all_rows = []
for run_id, label in [
    (RUN_ID_HYBRID_ZERO, "hybrid_zero_shot"),
    (RUN_ID_HYBRID_FEW, "hybrid_few_shot"),
    (RUN_ID_BASELINE_ZERO, "baseline_zero_shot"),
    (RUN_ID_BASELINE_FEW, "baseline_few_shot"),
]:
    row = collect_metric_row(run_id, label)
    if row:
        all_rows.append(row)

lines = [
    "# Notebook Execution Summary",
    "",
    "Generated by `notebooks/research_master_comprehensive.ipynb`.",
    "",
    "## Available Metrics",
]

if not all_rows:
    lines.append("No metrics JSON files found yet. Run evaluation cells first.")
else:
    table_df = pd.DataFrame(all_rows)
    lines.append(table_df.to_markdown(index=False))

summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Summary written to: {summary_path}")


## 8) Optional Sprint2 Summary Refresh

Aktifkan `RUN_SUMMARY_ONLY=True` jika ingin refresh agregasi sprint tanpa menjalankan preprocess/train/eval.


In [ ]:
if RUN_SUMMARY_ONLY:
    run_cmd_stream(
        [
            sys.executable,
            "scripts/research_sprint2.py",
            "--registry",
            str(REGISTRY_PATH),
            "--base-config",
            str(BASE_CONFIG_PATH),
            "--summarize-only",
        ],
        cwd=PROJECT_ROOT,
    )
else:
    print("RUN_SUMMARY_ONLY=False -> skip sprint summary refresh.")
